# Corpus build: HSBC and Barclays transcripts

104 earnings call transcripts, turned into `turns`, `exchanges` and `documents`. Every table is written as parquet and as CSV, so a consumer picks the format.

To run it anywhere else, set `DATA_INPUT` and `DATA_OUTPUT` in the Data paths cell and put the PDFs, with the manifest, in the input path. Nothing else is path dependent.

The scope for now is the preprocessing. `RUN_FINBERT` and `RUN_BERTOPIC` are both False in Common Variables, so the tone and topic sections below skip themselves and say so. Set either to True to run that half.


# How to run this notebook

Written for a Colab run. Locally there is nothing to do beyond pointing the two
paths in Data paths at the folders you already have.

## 1. Get the transcripts into your own Drive

They live in the team's shared Drive folder, 104 documents and 22 MB in total,
63 Barclays and 41 HSBC:

<https://drive.google.com/drive/folders/1yXh1lAdzHXCUNYWlf84EWhDiN1-w6lg-?usp=drive_link>

Open that link and add it to your own Drive: right click the folder, then
**Organise**, then **Add shortcut**. A shortcut is enough and takes no space. You
can copy the folder instead if you would rather have your own set.

This step is not optional. Colab mounts your Drive only, so a folder sitting in
**Shared with me** is invisible to the notebook, and that is what the shortcut
fixes.

What you point at has to hold `manifest_all.csv` alongside one folder per bank,
`hsbc/` and `barclays/`, with the quarter folders inside them. The manifest is the
list of documents this notebook works from, so the PDFs on their own are not
enough.

## 2. Mount the Drive

Use the folder icon in the left sidebar and **Mount Drive**, or run this in a cell
of its own:

```python
from google.colab import drive
drive.mount("/content/drive")
```

Approve the permission prompt. Your Drive then appears under
`/content/drive/MyDrive`, and the shortcut you made in step 1 is a folder inside it.

## 3. Point the two paths at it

In **Data paths** below, the cell holding nothing else, replace these two lines.
With a shortcut named `boe_transcripts` and a results folder named `boe_corpus`:

```python
DATA_INPUT = "/content/drive/MyDrive/boe_transcripts"
DATA_OUTPUT = "/content/drive/MyDrive/boe_corpus"
```

`DATA_INPUT` is read and never written to. `DATA_OUTPUT` does not have to exist
yet, the build creates it. Keep it in your own Drive rather than in the shared
folder, so you are not writing over everyone else's copy and so your tables are
still there tomorrow. `/content/boe_corpus` is faster and is thrown away when the
runtime disconnects.

Every other path in the notebook is derived from those two, so there is nothing
else to change.

## 4. Run it

**Runtime**, then **Run all**. The Setup cell above installs pdfplumber and nltk,
the build reads the 104 PDFs at roughly three seconds each, and the three tables
land in `DATA_OUTPUT` as both parquet and CSV. A `_checkpoint` folder appears
beside them holding one JSON file per document, which is what makes a second run
quick.

`RUN_FINBERT` and `RUN_BERTOPIC` are both False, so the tone and topic sections
skip themselves and say so. To run either one, set it to True, uncomment the
second install line in Setup, and switch the runtime to a GPU. Scoring the whole
corpus with FinBERT took about ten minutes on the machine this was built on, and a
Colab T4 will be slower.

## If something goes wrong

- **FileNotFoundError on `manifest_all.csv`**: `DATA_INPUT` is not the folder that
  holds the manifest. Check what is there with `!ls "$DATA_INPUT"`.
- **0 transcripts in scope**: the manifest was found but the bank folders are not
  beside it, so no row resolves to a file.
- **ValueError: No objects to concatenate**: the input holds too few documents for
  a single question and answer pair to be found. Copy the whole set rather than a
  handful of files.
- **The Drive asks to be mounted again**: Colab drops idle runtimes. Rerun the
  mount cell and carry on. Anything already extracted is read from the checkpoint
  rather than parsed twice.


## Setup


In [1]:
#importlib rather than the import cell below, because this runs before it
import importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    !pip install -q pdfplumber==0.11.10 nltk==3.9.4
    #and this line too, if RUN_FINBERT or RUN_BERTOPIC is set to True below
    #!pip install -q transformers==5.12.0 bertopic==0.17.4 sentence-transformers==5.5.1 umap-learn==0.5.12 hdbscan==0.8.44

Nothing to do outside Colab, where `google.colab` is absent and the pinned requirements are already installed. On Colab this pins the two libraries the extraction depends on, pdfplumber for reading the PDFs and nltk for the word list the ligature repair checks against. pandas, numpy, pyarrow and tqdm are left as Colab ships them, because pinning those forces a session restart for no gain. The commented line adds the model libraries, needed only if a run flag below is turned on.


# Common Functions

Every function the build needs is defined below, in the order the cells have to
run: a cell that calls a function comes after the cell defining it, so the
notebook runs top to bottom on a fresh kernel.


## Import Libraries


In [2]:
import csv
import difflib
import json
import os
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd

#libraries for reading the PDFs
import nltk
import pdfplumber
from nltk.corpus import words

#libraries for the build loop
from tqdm import tqdm

These are all the libraries the functions below need. Nothing else is imported, so the notebook runs on its own in Colab.


## Data paths


In [3]:
#The one input path, holding the manifest and one folder of PDFs per bank
DATA_INPUT = "data/raw"
#The one output path, holding every table the build writes. Created if it is missing
DATA_OUTPUT = "data/corpus"

These two are the only lines to change to run this somewhere else. DATA_INPUT holds the manifest and one folder of PDFs per bank, and is only ever read. DATA_OUTPUT is where every table is written, and is created if it is not there yet. On Colab both point at a mounted Drive folder. Every other path below is derived from them.


## Common Variables


In [4]:
#A line at the same page edge this often is furniture
HEADER_SHARE = 0.6
#A page number is a bare number, at most three digits
BARE_NUMBER = re.compile(r"\d{1,3}")
#A page number printed at the end of a footer line, "... London E14 5HP. 3"
#Without it the line differs on every page and never counts as furniture
TRAILING_NUMBER = re.compile(r"\s+\d{1,3}$")
#The legal notice Barclays appends after the call, as a line of its own
#62 Barclays transcripts carry it, 74k words glued onto the last answer
#2023 Q4 fixed income heads it "Disclaimer" one line above
NOTICE_HEADING = re.compile(r"\s*(important notice|disclaimer)\s*", re.I)
#fl before fi, because both are words and the corpus means flat
LIGATURES = ["fl", "fi", "ff", "ffi", "ffl"]
# nltk holds base forms only, so reflects and identified are both absent
SUFFIX_RULES = [("s", ""), ("es", ""), ("es", "e"), ("ed", ""), ("ed", "e"),
                ("ing", ""), ("ing", "e"), ("ied", "y"), ("ies", "y"),
                ("ly", ""), ("ion", "e"), ("ions", "e")]
#A word, letters plus the character a lost ligature leaves behind
WORD_OR_DAMAGED = re.compile("[A-Za-z�]+")
#nltk holds US base forms only, so a British spelling, a proper noun and a word
EXTRA_VOCAB = {"fulfil", "fulfils", "finalisation", "electronification",
               "brookfield", "cashflow", "cashflows", "flatline", "flatlining",
               "offload", "offloading"}
#NFKC not NFKD, because it recomposes and accented names stay one character
PUNCTUATION = {"\u2019": "'", "\u2018": "'", "\u201c": '"', "\u201d": '"',
               "\u2013": "-", "\u2014": "-", "\u2022": " ", "\u00a0": " "}
#Joined before the ligature repair, or the two halves never meet
DEHYPHEN = re.compile(r"(\w+)-[ \t]*\n[ \t]*(\w+)")
#Dots and apostrophes, dropped so C.S. and CS are the same speaker
_PUNCT = re.compile(r"[.\u2019']")
#GEORGES ELHEDERY, GROUP CEO: Welcome, all, to today's call.
#Accented capitals and commas in the affiliation are allowed, or PAUL
#FENNER-LEITÃO, SOCIÉTÉ GÉNÉRALE and CLAIRE BAIRD, CFO AND EXECUTIVE DIRECTOR,
#HSBC UK never enter the roster and every later turn of theirs is lost
HSBC_QUALIFIED = re.compile(
    r"^([A-ZÀ-Þ][A-ZÀ-Þ .'\-]{2,40}?), ([A-ZÀ-Þ][A-ZÀ-Þ .,&'\-/]{2,80}?):\s*(.*)$")
#GEORGES ELHEDERY: I'm going to take your first question on growth.
#A full stop for the colon once, RAUL SINHA. in 2024 Q2. Safe because a bare
#name must already be in the roster
HSBC_BARE = re.compile(r"^([A-ZÀ-Þ][A-ZÀ-Þ .'\-]{2,40}?)(?::\s*|\.\s+)(.*)$")
#HSBC misspells its own speakers (GEORGE ELHEDERY, EWEN STEVENSION, RAHUL for
#Raul Sinha). 0.85 matched all five seen and no heading
NAME_MATCH_CUTOFF = 0.85
#A name, allowing the initials Barclays uses: C.S. Venkatakrishnan
BARCLAYS_NAME = r"[A-Z][A-Za-z.'\-]*(?: [A-Z][A-Za-z.'\-]+){0,3}"
#A firm or job title, five words at most
BARCLAYS_AFFILIATION = r"[A-Z][\w.&'\-/]*(?: [\w.&'\-/]+){0,4}"
#Chris Hallam, Goldman Sachs
BARCLAYS_QUALIFIED = re.compile(rf"^({BARCLAYS_NAME}), ({BARCLAYS_AFFILIATION})$")
#Anna Cross
BARCLAYS_BARE = re.compile(rf"^({BARCLAYS_NAME})$")
#Grouped by firm, canonical spelling on the right. The comment on a group says
FIRM_SPELLINGS = {
    "Autonomous": ["Autonomous", "AUTONOMOUS", "AUTONOMOUS RESEARCH",
                   #AUTOMOUS is a typo in HSBC's 2023 Q3 fixed income transcript
                   "AUTOMOUS"],
    "Bank of America": ["Bank of America", "BANK OF AMERICA", "BofA"],
    "Barclays": ["BARCLAYS"],
    "Berenberg": ["Berenberg"],
    "Bloomberg Intelligence": ["Bloomberg Intelligence", "Bloomberg intelligence"],
    #Exane was bought outright by BNP Paribas in 2021 and rebranded, so the
    #plain and the joint name are one research house over this whole corpus
    "BNP Paribas Exane": ["Exane BNP Paribas", "BNP Paribas Exane",
                          "BNP PARIBAS EXANE", "BNP Exane", "EXANE",
                          "BNP Paribas", "BNP PARIBAS"],
    "BNY Newton": ["BNY Newton"],
    "China Securities": ["CHINA SECURITIES"],
    "CICC": ["CICC", "CHINA INTERNATIONAL CAPITAL CORPORATION"],
    #Citi is the brand Citigroup's research publishes under
    "Citi": ["Citi", "CITI", "Citigroup", "CITIGROUP", "Citi Bank"],
    "Credit Agricole": ["Credit Agricole"],
    "Credit Suisse": ["Credit Suisse", "CREDIT SUISSE"],
    "Deutsche Bank": ["Deutsche Bank", "DEUTSCHE BANK", "Deutsche"],
    #Deutsche Numis is the 2023 successor to Numis, kept apart from both parents
    "Deutsche Numis": ["DEUTSCHE NUMIS"],
    "HSBC": ["HSBC"],
    "Intesa Sanpaolo": ["Intesa Sanpaolo"],
    "Investec": ["Investec", "INVESTEC"],
    "Jefferies": ["Jefferies", "JEFFERIES",
                  #Jeferries lost its ff ligature and is not in the repair vocab
                  "Jeferries"],
    #Cazenove is the UK arm, and the analyst covering Barclays sits in it
    "JP Morgan": ["JP MORGAN", "JP Morgan", "J.P.Morgan", "JPMorgan", "JPM",
                  "J P MORGAN", "JP Morgan.", "JP MORGAN CAZENOVE"],
    "KBW": ["KBW"],
    "Kepler Cheuvreux": ["Kepler Cheuvreux", "Kepler Chevreux", "Kepler"],
    "Lazard AM": ["Lazard AM"],
    "MacKay Shields": ["MacKay Shields"],
    "Mediobanca": ["Mediobanca", "MEDIOBANCA"],
    "Morgan Stanley": ["Morgan Stanley", "MORGAN STANLEY"],
    "NatWest": ["Natwest"],
    "Numis": ["Numis", "NUMIS", "NUMIS SECURITIES"],
    "Peel Hunt": ["Peel Hunt"],
    "PGIM": ["PGIM"],
    "RBC": ["RBC", "RBC CAPITAL MARKETS", "RBC EQUITIES EUROPE"],
    #Redburn traded as Rothschild & Co Redburn and then Redburn Atlantic over
    #this period. The short name is the one that spans all five years
    "Redburn": ["Redburn", "REDBURN", "Redburn Atlantic", "REDBURN ATLANTIC",
                "ROTHSCHILD & CO REDBURN", "Rothschild & Co Redburn",
                "Rothschild and Co Redburn"],
    "Shore Capital": ["Shore Capital"],
    "Societe Generale": ["Societe Generale", "Société Générale"],
    "T. Rowe Price": ["T. ROWE PRICE", "T ROWE PRICE"],
    "UBS": ["UBS"],
    "Verition": ["VERITION", "Verition", "Verition fund management"],
    "ACR": ["ACR"],
    "DBS": ["DBS"],
    "Allspring Global Investments": ["Allspring Global Invesments"],
    "Goldman Sachs": ["Goldman Sachs", "GOLDMAN SACHS"],
}
#Group roles, in either bank's wording. Checked before the analyst default
MANAGEMENT = ("group ceo", "group cfo", "chief executive", "chief financial",
              "finance director", "financial director", "financial officer",
              "group chief", "chairman", "treasurer", "head of", "director of",
              "managing director", "group treasurer", "cfo", "ceo",
              "financial controller", "accounting officer")
#A speaker whose affiliation names the bank itself is not an analyst
#Only the bank holding the call counts. Matching both put Robin Down of HSBC
#on Barclays calls and Aman Rakkar of Barclays on HSBC calls in management
OWN_BANK = ("barclays", "hsbc")
#IR wording, checked before the analyst default
INVESTOR_RELATIONS = ("investor relations", "head of ir")
#The call operator, who is neither management nor an analyst
OPERATOR = ("operator", "moderator")
#Both banks name the audience in the filename
FIXED_INCOME = re.compile(r"fixed[\-_ ]?income", re.I)
#A management speech, with no Q&A anywhere in the document
PREPARED_ONLY = re.compile(r"management[\-_ ](speech|script)", re.I)
#A Q&A transcript published on its own, with no prepared remarks
QA_ONLY = re.compile(r"q[\-_ ]?and[\-_ ]?a|q[\-_ ]a[\-_ ]|sellside", re.I)
#One JSON file per document, so reparsing costs nothing
CHECKPOINT = f"{DATA_OUTPUT}/_checkpoint"
#The manifest the downloader wrote, one row per document it fetched
MANIFEST = f"{DATA_INPUT}/manifest_all.csv"
#The three tables the build produces, named here so the driver can read them back
TURNS_FILE = f"{DATA_OUTPUT}/turns.parquet"
EXCHANGES_FILE = f"{DATA_OUTPUT}/exchanges.parquet"
DOCUMENTS_FILE = f"{DATA_OUTPUT}/documents.parquet"
#The two banks the corpus is narrowed to, 76% of the transcript text
BANKS = ("hsbc", "barclays")
# If true, every document is parsed again, not read from the checkpoint
FORCE_REBUILD = False
#If true, the three tables are built again rather than read back from parquet
REBUILD_CORPUS = False
#The column order turns.parquet ships in
TURN_COLUMNS = ["turn_id", "doc_sha256", "bank", "fiscal_year", "fiscal_quarter",
                "audience", "doc_format", "section", "speaker", "speaker_norm",
                "affiliation", "role", "page_start", "turn_idx", "n_words",
                "n_chars", "text", "repaired"]
#The column order exchanges.parquet ships in
EXCHANGE_COLUMNS = ["exchange_id", "doc_sha256", "bank", "period", "audience",
                    "analyst", "analyst_firm", "question_text", "question_turn_id",
                    "question_page", "answer_text", "answer_turn_ids", "responder_names",
                    "responder_roles", "n_words_q", "n_words_a"]
#The column order documents.parquet ships in
DOCUMENT_COLUMNS = ["doc_sha256", "relpath", "bank", "fiscal_year",
                    "fiscal_quarter", "audience", "doc_format", "source_url",
                    "source_page", "fetched_at", "n_pages",
                    "n_turns", "n_exchanges", "n_speakers", "fffd_before",
                    "silent_repairs", "parse_status", "warnings"]
#Where the scored turns are written, and read back from on a later run
SENTIMENT_FILE = f"{DATA_OUTPUT}/sentiment.parquet"
#The same scores rolled up to the exchange they belong to
SENTIMENT_EXCHANGES_FILE = f"{DATA_OUTPUT}/sentiment_exchanges.parquet"
#If true, every turn is scored again. Ten minutes for 3,526 turns on the 4070
RESCORE_SENTIMENT = False
#If false, FinBERT is skipped altogether and no sentiment table is written
RUN_FINBERT = True
#The finance-tuned model the tone scores come from
FINBERT = "ProsusAI/finbert"
#A general sentiment model, scored alongside to catch FinBERT drifting
BASELINE = "distilbert-base-uncased-finetuned-sst-2-english"
#FinBERT's wordpiece limit, past which it truncates without saying so
MAX_TOKENS = 512
#Sentence end, with lookbehinds
ABBREV = r"(?<!\bMr)(?<!\bMrs)(?<!\bDr)(?<!\bSt)(?<!\bInc)(?<!\bLtd)(?<!\bplc)(?<!\bNo)"
#A sentence ends at . ! or ? followed by a space and a capital
SENTENCE_END = re.compile(rf"{ABBREV}(?<![A-Z0-9])([.!?])\s+(?=[A-Z])")
#A hard ceiling in words, well under the 512 wordpiece limit
MAX_SENTENCE_WORDS = 300
#The sentence embedder BERTopic uses, small enough for the 4070
EMBEDDINGS = "BAAI/bge-small-en-v1.5"
#Where the fitted topic model is saved between runs
MODEL_DIR = f"{DATA_OUTPUT}/topic_model"
#One row per analyst question, and one per topic the model found
TOPICS_FILE = f"{DATA_OUTPUT}/topics.parquet"
TOPIC_INFO_FILE = f"{DATA_OUTPUT}/topic_info.parquet"
#The exchange_id of every question the model was fitted on, in fit order
FIT_IDS_FILE = "fit_exchange_ids.json"
# If true, the model is fitted again. This renumbers every topic
REFIT_BERTOPIC = False
#If false, BERTopic is skipped altogether and no topic table is written
RUN_BERTOPIC = True
#Pinned once and passed everywhere, so a rerun gives the same topics
SEED = 42
#The politeness an analyst opens with, which is not what the question is about
BOILERPLATE = re.compile(
    r"\b(thanks?( you)?( very much)?( for taking my questions?)?|"
    r"good (morning|afternoon|evening)( everybody| everyone| both| all)?|"
    r"two questions from me|a couple of questions|just a quick one|"
    r"if i (may|could)|morning both|can you hear me|first of all)\b[,.]?\s*",
    re.I)
#The silent form of the defect
SILENT_CORRUPTION = re.compile(
    r"\b\w*(?:diferen|ofer|ofset|efect|afect|signiicant|eficien)\w*\b")
#One person talking is not a runaway turn, so these are exempt
MONOLOGUE_FORMATS = ("prepared_only",)
#Longer than this and a turn is two speakers glued together
MAX_TURN_WORDS = 5000

These are the constants every function below reads. Changing a threshold is done here rather than hunting through the code. The run flags sit here too: RUN_FINBERT and RUN_BERTOPIC are both False, which is what keeps this run to the preprocessing.


## Page layout


In [5]:
def strip_running_headers(pages):
    edge_line_counts = {}
    for page in pages:
        page_lines = [l.strip() for l in page.splitlines() if l.strip()]
        for line in set(page_lines[:1] + page_lines[-1:]):
            line = TRAILING_NUMBER.sub("", line)
            edge_line_counts[line] = edge_line_counts.get(line, 0) + 1

    #Speaker names repeat too, but never at the page edge
    furniture = {line for line, n in edge_line_counts.items()
                 if n >= len(pages) * HEADER_SHARE}

    pages_without_headers, n_removed = [], 0
    for page in pages:
        page_lines = page.splitlines()
        kept_lines = []
        for i, line in enumerate(page_lines):
            at_page_edge = i == 0 or i == len(page_lines) - 1
            if at_page_edge and TRAILING_NUMBER.sub("", line.strip()) in furniture:
                n_removed += 1
                continue
            kept_lines.append(line)
        pages_without_headers.append("\n".join(kept_lines))
    return pages_without_headers, n_removed

This function is used to remove the header and footer both banks print on every page, found by position rather than by a list.


In [6]:
def strip_page_numbers(pages):
    pages_without_numbers, n_removed = [], 0
    for page in pages:
        page_lines = page.splitlines()
        kept_lines = []
        for i, line in enumerate(page_lines):
            # A bare number mid paragraph is somebody saying it out loud
            at_page_edge = i <= 1 or i >= len(page_lines) - 2
            if at_page_edge and BARE_NUMBER.fullmatch(line.strip()):
                n_removed += 1
                continue
            kept_lines.append(line)
        pages_without_numbers.append("\n".join(kept_lines))
    return pages_without_numbers, n_removed

This function is used to drop page numbers, only at the top or bottom of a page, because a bare number mid paragraph is somebody saying it out loud.


In [7]:
def strip_closing_notice(pages):
    for page_idx, page in enumerate(pages):
        page_lines = page.splitlines()
        for line_idx, line in enumerate(page_lines):
            if NOTICE_HEADING.fullmatch(line):
                removed = page_lines[line_idx:] + [l for p in pages[page_idx + 1:]
                                                   for l in p.splitlines()]
                #Blanked rather than dropped, so n_pages stays the PDF's page count
                kept_pages = (pages[:page_idx] + ["\n".join(page_lines[:line_idx])]
                              + [""] * (len(pages) - page_idx - 1))
                return kept_pages, sum(1 for l in removed if l.strip())
    return pages, 0

This function is used to cut the legal notice Barclays appends after the call, from its heading to the end of the document. The pages are blanked rather than dropped, so the page count stays the PDF's.


## Text extraction


In [8]:
#Replace pypdf with pdfplumber, which broke 12% of the Barclays tokens into
#single letters. Three times slower and worth it
def read_pdf_pages(path):
    if not os.path.isfile(path):
        raise ValueError(f"Not a file: {path}")

    with pdfplumber.open(path) as pdf:
        return [page.extract_text() or "" for page in pdf.pages]

This function is used to read a PDF into one string per page, which is what lets a number be traced back to the page it came from.


In [9]:
#nltk lists ofer, so damaged offer would survive gate 4
try:
    VOCAB = ({w.lower() for w in words.words()} | EXTRA_VOCAB) - {"ofer"}
except LookupError:
    nltk.download("words")
    VOCAB = ({w.lower() for w in words.words()} | EXTRA_VOCAB) - {"ofer"}

The nltk word list, used to decide whether a repaired word is real. It downloads itself the first time the notebook runs.


In [10]:
def is_word(candidate):
    candidate = candidate.lower()
    if candidate in VOCAB:
        return True

    for suffix, ending in SUFFIX_RULES:
        if candidate.endswith(suffix):
            stem = candidate[:-len(suffix)] + ending
            if stem in VOCAB:
                return True
            #offsetting doubles the last letter of offset
            if len(stem) > 2 and stem[-1] == stem[-2] and stem[:-1] in VOCAB:
                return True
    return False

This function is used to check a word against the dictionary, reducing it to a base form first, because nltk holds no plurals or past tenses.


In [11]:
def repair_ligatures(text):
    repaired_parts = []
    n_repairs = 0
    copied_upto = 0
    for match in WORD_OR_DAMAGED.finditer(text):
        word = match.group(0)
        if "�" in word:
            candidates = [word.replace("�", lig, 1) for lig in LIGATURES]
        elif "f" in word and not is_word(word):
            #The surviving f says where the other one was. Every word-initial
            #case was "F inancial", a split word, not a lost ligature
            candidates = [word[:i] + "f" + word[i:]
                          for i, c in enumerate(word) if c == "f"]
        else:
            continue

        repaired_word = next((c for c in candidates if is_word(c)), None)
        if repaired_word is None:
            continue

        repaired_parts.append(text[copied_upto:match.start()])
        repaired_parts.append(repaired_word)
        copied_upto = match.end()
        n_repairs += 1

    repaired_parts.append(text[copied_upto:])
    return "".join(repaired_parts), n_repairs

This function is used to repair words the PDF font broke, like difficult arriving as dificult, by trying each ligature and keeping the one that makes a real word.


In [12]:
#Drop the hyphen only when the halves make a word
#Otherwise cost-effectively joins into costeffectively. The repair is tried
#too, so efi-ciency still joins
def _join_broken_word(match):
    left, right = match.group(1), match.group(2)
    joined_word = left + right
    if is_word(joined_word) or is_word(repair_ligatures(joined_word)[0]):
        return joined_word
    return f"{left}-{right}"

This function is used to join a word split across a line break, but only when the two halves make a real word.


In [13]:
def normalise_text(text):
    text = unicodedata.normalize("NFKC", text)
    for bad, good in PUNCTUATION.items():
        text = text.replace(bad, good)
    text = DEHYPHEN.sub(_join_broken_word, text)

    # Collapse spaces but keep newlines, segmentation reads the line starts
    text = re.sub(r"[^\S\n]+", " ", text)
    return "\n".join(line.strip() for line in text.split("\n"))

This function is used to fold the typographic punctuation and collapse spaces, keeping newlines, because segmentation reads speaker names off the start of a line.


In [14]:
#The order is load-bearing. Furniture first, so no header is
#glued onto a sentence, then de-hyphenation, and the repair last
#because it needs whole words
def extract_document(path):
    raw_pages = read_pdf_pages(path)
    #Peeled until nothing comes off, because one pass takes one edge line and
    #a page number hid the three line Barclays PRA footer under it
    pages, n_headers, n_numbers = raw_pages, 0, 0
    while True:
        pages, n_headers_here = strip_running_headers(pages)
        pages, n_numbers_here = strip_page_numbers(pages)
        n_headers += n_headers_here
        n_numbers += n_numbers_here
        if not n_headers_here and not n_numbers_here:
            break
    #Empty pages are counted before the notice is blanked, so the stat still
    #flags a page the extractor failed on
    n_empty_pages = sum(1 for page in pages if not page.strip())
    pages, n_notice = strip_closing_notice(pages)

    fffd_before = sum(page.count("�") for page in raw_pages)
    clean_pages, n_repairs = [], 0
    for page in pages:
        page = normalise_text(page)
        page, n_repaired_here = repair_ligatures(page)
        clean_pages.append(page)
        n_repairs += n_repaired_here

    stats = {
        "n_pages": len(clean_pages),
        "n_chars": sum(len(page) for page in clean_pages),
        "n_empty_pages": n_empty_pages,
        "headers_removed": n_headers,
        "page_numbers_removed": n_numbers,
        "notice_lines_removed": n_notice,
        "fffd_before": fffd_before,
        "fffd_after": sum(page.count("�") for page in clean_pages),
        "silent_repairs": n_repairs,
    }
    return clean_pages, stats

This function is used to run the whole extraction for one document and report what it had to repair.


## Speaker names


In [15]:
def normalise_name(name):
    return " ".join(_PUNCT.sub("", name or "").lower().split())

This function is used to reduce a name to a comparable form, so C.S. Venkatakrishnan and CS Venkatakrishnan are the same speaker.


## HSBC speaker lines


In [16]:
def match_speaker_line_hsbc(line, roster):
    line = line.strip()

    m = HSBC_QUALIFIED.match(line)
    if m:
        return m.group(1).title(), m.group(2), m.group(3)

    m = HSBC_BARE.match(line)
    if m:
        #Roster names only, or every shouted heading becomes a speaker
        name = normalise_name(m.group(1))
        known = roster.get(name)
        if not known:
            close = difflib.get_close_matches(name, roster, n=1, cutoff=NAME_MATCH_CUTOFF)
            known = roster[close[0]] if close else None
        if known:
            return known[0], known[1], m.group(2)
    return None

This function is used to recognise an HSBC speaker line, which is written in capitals with the job title after the name.


## Barclays speaker lines


In [17]:
def match_speaker_line_barclays(line, roster):
    line = line.strip()

    m = BARCLAYS_QUALIFIED.match(line)
    if m:
        return m.group(1), m.group(2), ""

    m = BARCLAYS_BARE.match(line)
    if m:
        known = roster.get(normalise_name(m.group(1)))
        if known:
            return known[0], known[1], ""
    return None

This function is used to recognise a Barclays speaker line, which is written in normal case with the firm after the name.


## Analyst firms


In [18]:
def firm_key(affiliation):
    decomposed = unicodedata.normalize("NFKD", affiliation or "")
    without_accents = "".join(c for c in decomposed
                              if not unicodedata.combining(c))
    #Ampersand and "and" are the same word in Rothschild & Co Redburn
    folded = without_accents.lower().replace("&", " and ")
    folded = re.sub(r"\band\b", " ", folded)
    return " ".join(re.sub(r"[^a-z0-9 ]+", " ", folded).split())

This function is used to reduce a firm name to a comparable key, dropping accents, case, punctuation and the word and, so Rothschild & Co Redburn and Rothschild and Co Redburn give the same key.


In [19]:
FIRM_ALIASES = {firm_key(spelling): canonical
                for canonical, spellings in FIRM_SPELLINGS.items()
                for spelling in spellings}

The lookup from every known spelling of a firm, reduced to its key, to the one name the tables use.


In [20]:
def normalise_firm(affiliation):
    if not affiliation or not affiliation.strip():
        return affiliation
    return FIRM_ALIASES.get(firm_key(affiliation), affiliation.strip())

This function is used to give each analyst firm its canonical name, so one firm written several ways across calls is counted as one firm. A spelling not in the list is kept as written, only trimmed.


## Speaker roster


In [21]:
MATCHERS = {"hsbc": match_speaker_line_hsbc, "barclays": match_speaker_line_barclays}

The two matchers, picked by bank. This one cannot sit with the other constants because it names the functions above it.


In [22]:
def classify_role(affiliation, bank=None):
    wording = (affiliation or "").lower()
    own_bank = (bank,) if bank else OWN_BANK
    if any(k in wording for k in OPERATOR):
        return "operator"
    if any(k in wording for k in INVESTOR_RELATIONS):
        return "ir"
    if any(k in wording for k in MANAGEMENT) or any(k in wording for k in own_bank):
        return "management"
    if not wording.strip():
        return "unknown"
    #anything left is an outside firm, so an analyst
    return "analyst"

This function is used to decide whether a speaker is management, an analyst, the operator or investor relations, from their affiliation alone.


In [23]:
def find_speaker_candidates(lines, bank):
    match_speaker_line = MATCHERS[bank]
    roster, candidates = {}, []
    for line_no, line in enumerate(lines):
        matched = match_speaker_line(line, roster)
        if not matched:
            continue
        speaker, affiliation, rest = matched
        key = normalise_name(speaker)
        if key not in roster:
            # The first qualified appearance wins, a later bare match must not
            # overwrite the spelling it taught
            roster[key] = (speaker, affiliation)
        known_spelling, known_affiliation = roster[key]
        candidates.append({"line_no": line_no, "speaker": known_spelling,
                           "affiliation": affiliation or known_affiliation,
                           "rest": rest})
    return candidates

This function is used to find every line that opens a turn, learning the canonical spelling of each name the first time it appears in full.


In [24]:
def build_roster(candidates, bank=None):
    speaker_rows = {}
    for candidate in candidates:
        name = candidate["speaker"]
        if name not in speaker_rows:
            speaker_rows[name] = {
                "speaker": name,
                "speaker_norm": normalise_name(name).replace(" ", "_"),
                "affiliation": candidate["affiliation"],
                "n_turns": 0,
                "first_line": candidate["line_no"],
            }
        speaker_record = speaker_rows[name]
        #A bare match has no affiliation, never blank a known one
        if candidate["affiliation"] and not speaker_record["affiliation"]:
            speaker_record["affiliation"] = candidate["affiliation"]
        speaker_record["n_turns"] += 1

    df_roster = pd.DataFrame(speaker_rows.values())
    if df_roster.empty:
        return pd.DataFrame(columns=["speaker", "speaker_norm", "affiliation",
                                     "role", "n_turns", "first_line"])
    df_roster["role"] = df_roster.affiliation.map(lambda a: classify_role(a, bank))
    return df_roster.sort_values("first_line").reset_index(drop=True)

This function is used to turn those candidates into one row per speaker, with their role and how many turns they took.


## Segmentation


In [25]:
def classify_audience(relpath, link_text=""):
    filename_and_link = f"{relpath} {link_text}"
    audience = ("fixed_income" if FIXED_INCOME.search(filename_and_link)
                else "equity")

    if PREPARED_ONLY.search(filename_and_link):
        doc_format = "prepared_only"
    elif QA_ONLY.search(filename_and_link):
        doc_format = "qa_only"
    else:
        #HSBC publishes prepared remarks and Q&A as one document
        doc_format = "prepared_and_qa"
    return audience, doc_format

This function is used to read the audience and the document format off the filename, since both banks put them there.


In [26]:
def segment_turns(pages, df_roster, bank):
    match_speaker_line = MATCHERS[bank]
    roster = {normalise_name(row.speaker): (row.speaker, row.affiliation)
              for row in df_roster.itertuples()}

    turns = []
    for page_no, page in enumerate(pages, 1):
        for line in page.splitlines():
            if not line.strip():
                continue
            matched = match_speaker_line(line, roster)
            if matched:
                speaker, affiliation, rest = matched
                #A line that names the firm itself is normalised here too, or a
                #speaker introduced mid document keeps the printed spelling
                affiliation = normalise_firm(affiliation)
                turns.append({"speaker": speaker, "affiliation": affiliation,
                              "role": classify_role(affiliation, bank),
                              "page_start": page_no,
                              "lines": [rest] if rest.strip() else []})
            elif turns:
                turns[-1]["lines"].append(line.strip())
            # Lines before the first speaker are the cover page, dropped

    turn_rows = []
    for turn_idx, turn in enumerate(turns):
        turn_text = " ".join(" ".join(turn["lines"]).split())
        if not turn_text:
            continue
        turn_rows.append({
            "turn_idx": turn_idx,
            "speaker": turn["speaker"],
            "speaker_norm": normalise_name(turn["speaker"]).replace(" ", "_"),
            "affiliation": turn["affiliation"],
            "role": turn["role"],
            "page_start": turn["page_start"],
            "n_words": len(turn_text.split()),
            "n_chars": len(turn_text),
            "text": turn_text,
        })
    return pd.DataFrame(turn_rows)

This function is used to split a transcript into turns, one per speaker, keeping the page each one started on.


In [27]:
def label_sections(df_turns, doc_format):
    df_labelled = df_turns.copy()
    if df_labelled.empty:
        df_labelled["section"] = []
        return df_labelled

    if doc_format == "prepared_only":
        df_labelled["section"] = "prepared"
        return df_labelled
    if doc_format == "qa_only":
        df_labelled["section"] = "qa"
        return df_labelled

    #The Q&A starts at the first turn from an outside firm
    #The narrative cue is worded differently every quarter
    analyst_turns = df_labelled.index[df_labelled.role == "analyst"]
    first_analyst_turn = (analyst_turns[0] if len(analyst_turns)
                          else len(df_labelled))
    df_labelled["section"] = ["prepared" if i < first_analyst_turn else "qa"
                              for i in range(len(df_labelled))]
    return df_labelled

This function is used to mark each turn as prepared remarks or Q&A, the boundary being the first analyst question.


In [28]:
def build_exchanges(df_turns):
    qa_turns = df_turns[df_turns.section == "qa"].reset_index(drop=True)
    exchange_rows = []
    for position, question_turn in qa_turns.iterrows():
        if question_turn.role != "analyst":
            continue
        answer_turns = []
        for _, following_turn in qa_turns.loc[position + 1:].iterrows():
            if following_turn.role == "analyst":
                break
            answer_turns.append(following_turn)
        if not answer_turns:
            continue
        answer_text = " ".join(turn.text for turn in answer_turns)
        exchange_rows.append({
            "exchange_idx": len(exchange_rows),
            "analyst": question_turn.speaker,
            "analyst_firm": question_turn.affiliation,
            "question_text": question_turn.text,
            "question_turn_idx": int(question_turn.turn_idx),
            "answer_text": answer_text,
            "answer_turn_idx": [int(turn.turn_idx) for turn in answer_turns],
            "responder_names": [turn.speaker for turn in answer_turns],
            "responder_roles": [turn.role for turn in answer_turns],
            "n_words_q": int(question_turn.n_words),
            "n_words_a": len(answer_text.split()),
        })
    return pd.DataFrame(exchange_rows)

This function is used to pair each analyst question with the answers that followed it, which is the grain the comparison work needs.


## Build loop


In [29]:
def build_documents(manifest_path, banks=BANKS):
    document_rows = []
    with open(manifest_path, encoding="utf-8") as fh:
        for manifest_row in csv.DictReader(fh):
            if (manifest_row["bank"] not in banks
                    or manifest_row["doctype"] != "transcript"):
                continue
            audience, doc_format = classify_audience(manifest_row["relpath"],
                                                     manifest_row["link_text"])
            document_rows.append({
                "doc_sha256": manifest_row["sha256"],
                "relpath": manifest_row["relpath"],
                "bank": manifest_row["bank"],
                "fiscal_year": int(manifest_row["fiscal_year"]),
                "fiscal_quarter": int(manifest_row["fiscal_quarter"]),
                "audience": audience,
                "doc_format": doc_format,
                #Carried from the manifest so a flagged row cites its URL and page
                "source_url": manifest_row["source_url"],
                "source_page": manifest_row["source_page"],
                "fetched_at": manifest_row["fetched_at"],
            })
    return pd.DataFrame(document_rows).sort_values(
        ["bank", "fiscal_year", "fiscal_quarter", "relpath"]).reset_index(drop=True)

This function is used to read the manifest, the CSV the downloader wrote with one row per document it fetched, and list the transcripts in scope, which is HSBC and Barclays only.


In [30]:
def load_pages(doc, root=DATA_INPUT, force_rebuild=FORCE_REBUILD):
    os.makedirs(CHECKPOINT, exist_ok=True)
    checkpoint_path = os.path.join(CHECKPOINT, f"{doc['doc_sha256'][:12]}.json")
    if os.path.exists(checkpoint_path) and not force_rebuild:
        checkpointed = json.load(open(checkpoint_path, encoding="utf-8"))
        return checkpointed["pages"], checkpointed["stats"]

    pages, stats = extract_document(os.path.join(root, doc["relpath"]))
    with open(checkpoint_path, "w", encoding="utf-8") as fh:
        json.dump({"pages": pages, "stats": stats}, fh)
    return pages, stats

This function is used to extract one document, caching the pages so that changing the segmentation later costs nothing.


In [31]:
def build_bank_rosters(df_documents, root=DATA_INPUT, force_rebuild=FORCE_REBUILD):
    rosters = {}
    for doc in tqdm(df_documents.to_dict("records"), desc="pass 1: rosters"):
        pages, _ = load_pages(doc, root, force_rebuild)
        document_lines = [line for page in pages for line in page.splitlines()
                          if line.strip()]
        for candidate in find_speaker_candidates(document_lines, doc["bank"]):
            if candidate["affiliation"]:
                rosters.setdefault(doc["bank"], {}).setdefault(
                    candidate["speaker"], candidate["affiliation"])

    return {bank: build_roster([{"line_no": 0, "speaker": s, "affiliation": a}
                                for s, a in names.items()], bank)
            for bank, names in rosters.items()}

This function is used to pool a bank's speakers across every document it published, because a speaker is only introduced in full occasionally.


In [32]:
def parse_document(doc, root=DATA_INPUT, bank_roster=None):
    pages, stats = load_pages(doc, root)
    document_lines = [line for page in pages for line in page.splitlines()
                      if line.strip()]
    roster = build_roster(find_speaker_candidates(document_lines, doc["bank"]),
                          doc["bank"])
    if bank_roster is not None and len(bank_roster):
        #The document's own introductions win, the roster fills the gaps
        named_here = set(roster.speaker) if len(roster) else set()
        from_other_documents = bank_roster[~bank_roster.speaker.isin(named_here)]
        roster = pd.concat([roster, from_other_documents], ignore_index=True)
    turns = label_sections(segment_turns(pages, roster, doc["bank"]),
                           doc["doc_format"])
    exchanges = build_exchanges(turns) if len(turns) else pd.DataFrame()

    doc_sha = doc["doc_sha256"]
    if len(turns):
        turns = turns.assign(
            turn_id=[f"{doc_sha[:12]}-{i:04d}" for i in turns.turn_idx],
            doc_sha256=doc_sha, bank=doc["bank"],
            fiscal_year=doc["fiscal_year"], fiscal_quarter=doc["fiscal_quarter"],
            audience=doc["audience"], doc_format=doc["doc_format"],
            repaired=stats["silent_repairs"] > 0)
    if len(exchanges):
        turn_id_by_idx = dict(zip(turns.turn_idx, turns.turn_id))
        page_by_idx = dict(zip(turns.turn_idx, turns.page_start))
        exchanges = exchanges.assign(
            exchange_id=[f"{doc_sha[:12]}-x{i:03d}" for i in exchanges.exchange_idx],
            doc_sha256=doc_sha, bank=doc["bank"],
            period=f"{doc['fiscal_year']}_Q{doc['fiscal_quarter']}",
            audience=doc["audience"],
            question_turn_id=[turn_id_by_idx[i]
                              for i in exchanges.question_turn_idx],
            question_page=[int(page_by_idx[i]) for i in exchanges.question_turn_idx],
            answer_turn_ids=[[turn_id_by_idx[i] for i in ids]
                             for ids in exchanges.answer_turn_idx])

    #No Q&A is not broken, it is a management speech
    parse_status = "ok"
    if not len(turns):
        parse_status = "failed"
    elif doc["doc_format"] != "prepared_only" and not len(exchanges):
        parse_status = "no_qa"
    elif doc["doc_format"] != "prepared_only" and len(turns) < 5:
        parse_status = "low_turns"

    document_summary = {
        **doc, **stats, "n_turns": len(turns), "n_exchanges": len(exchanges),
        "n_speakers": int(turns.speaker.nunique()) if len(turns) else 0,
        "parse_status": parse_status, "warnings": ""}
    return turns, exchanges, document_summary

This function is used to take one document all the way from PDF to turns and exchanges, and report how it went.


In [33]:
def build_corpus(df_documents, force_rebuild=FORCE_REBUILD, root=DATA_INPUT):
    print("pass 1: extracting and pooling each bank's roster", flush=True)
    rosters = build_bank_rosters(df_documents, root, force_rebuild)
    for bank, df_bank_roster in rosters.items():
        print(f"  {bank}: {len(df_bank_roster)} speakers, "
              f"{int((df_bank_roster.role == 'management').sum())} management",
              flush=True)

    all_turns, all_exchanges, document_summaries = [], [], []
    for doc in tqdm(df_documents.to_dict("records"), desc="pass 2: segmenting"):
        try:
            turns, exchanges, document_summary = parse_document(
                doc, root, rosters.get(doc["bank"]))
        except Exception as failure:
            document_summary = {
                **doc, "n_turns": 0, "n_exchanges": 0, "n_speakers": 0,
                "parse_status": "failed",
                "warnings": f"{type(failure).__name__}: {failure}"}
            turns = exchanges = pd.DataFrame()
            #tqdm.write, not print, or the failure lands on top of the bar
            tqdm.write(f"  ! {doc['relpath'][:44]}: {failure}")
        all_turns.append(turns)
        all_exchanges.append(exchanges)
        document_summaries.append(document_summary)

    df_turns = pd.concat([df for df in all_turns if len(df)], ignore_index=True)
    df_exchanges = pd.concat([df for df in all_exchanges if len(df)],
                             ignore_index=True)
    df_documents = pd.DataFrame(document_summaries)
    return df_turns, df_exchanges, df_documents

This function is used to run the whole build over every document, in two passes, the first to learn the rosters.


In [34]:
def write_table(df_table, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    csv_path = os.path.splitext(path)[0] + ".csv"
    df_table.to_parquet(path, index=False)
    df_table.to_csv(csv_path, index=False)
    return path, csv_path

This function is used to write one table twice, parquet and CSV side by side, so a consumer reads whichever suits them. A list column arrives in the CSV as its text form.


In [35]:
def write_corpus(df_turns, df_exchanges, df_documents):
    os.makedirs(DATA_OUTPUT, exist_ok=True)
    written_tables = {}
    for name, df_table, columns, table_path in (
            ("turns", df_turns, TURN_COLUMNS, TURNS_FILE),
            ("exchanges", df_exchanges, EXCHANGE_COLUMNS, EXCHANGES_FILE),
            ("documents", df_documents, DOCUMENT_COLUMNS, DOCUMENTS_FILE)):
        present = [c for c in columns if c in df_table.columns]
        write_table(df_table[present], table_path)
        written_tables[name] = (table_path, len(df_table))
    return written_tables

This function is used to write the three tables in a fixed column order, each one as parquet and as CSV.


In [36]:
def build_or_load_corpus(rebuild=REBUILD_CORPUS):
    tables = (TURNS_FILE, EXCHANGES_FILE, DOCUMENTS_FILE)
    if all(os.path.isfile(table_path) for table_path in tables) and not rebuild:
        loaded = [pd.read_parquet(table_path) for table_path in tables]
        print(f"loaded the tables from {DATA_OUTPUT} "
              f"(set REBUILD_CORPUS=True to build them again)")
        return loaded[0], loaded[1], loaded[2], False

    df_documents = build_documents(MANIFEST)
    print(f"{len(df_documents)} transcripts in scope")
    df_turns, df_exchanges, df_document_summaries = build_corpus(df_documents)
    written_tables = write_corpus(df_turns, df_exchanges, df_document_summaries)
    for name, (table_path, n_rows) in written_tables.items():
        print(f"  wrote {table_path}  {n_rows} rows")
    return df_turns, df_exchanges, df_document_summaries, True

This function is used to read the three tables back from parquet, or build them from the PDFs the first time.


In [37]:
def main():
    build_or_load_corpus(rebuild=True)
    return 0

This function is used to run the build from the command line. In the notebook the driver below does the same thing.


## Sentiment


In [38]:
def split_sentences(text):
    candidates = SENTENCE_END.sub(r"\1\n", text or "").split("\n")
    sentences = []
    for candidate in candidates:
        candidate = candidate.strip()
        if len(candidate) <= 2:
            continue
        candidate_words = candidate.split()
        if len(candidate_words) <= MAX_SENTENCE_WORDS:
            sentences.append(candidate)
            continue
        for start in range(0, len(candidate_words), MAX_SENTENCE_WORDS):
            sentences.append(" ".join(
                candidate_words[start:start + MAX_SENTENCE_WORDS]))
    return sentences

This function is used to split a turn into sentences, keeping figures like 11.5% and Basel 3.1 in one piece.


In [39]:
def fit_to_budget(sentences, tokeniser, limit=MAX_TOKENS):
    if tokeniser is None:
        return sentences, [0] * len(sentences)

    within_budget, token_counts = [], []
    still_to_measure = list(sentences)
    while still_to_measure:
        piece = still_to_measure.pop(0)
        n_tokens = len(tokeniser.encode(piece))
        piece_words = piece.split()
        if n_tokens <= limit or len(piece_words) < 2:
            within_budget.append(piece)
            token_counts.append(n_tokens)
            continue
        halfway = len(piece_words) // 2
        still_to_measure.insert(0, " ".join(piece_words[halfway:]))
        still_to_measure.insert(0, " ".join(piece_words[:halfway]))
    return within_budget, token_counts

This function is used to split anything the tokeniser says is over budget, measured rather than guessed.


In [40]:
def score_turn(text, finbert, baseline, tokeniser=None):
    sentences = split_sentences(text)
    if not sentences:
        return None
    sentences, n_tokens = fit_to_budget(sentences, tokeniser)

    sentence_weights = [len(s.split()) for s in sentences]
    total_weight = sum(sentence_weights) or 1
    weighted_scores = {"positive": 0.0, "negative": 0.0, "neutral": 0.0}
    for predictions, weight in zip(
            finbert(sentences, truncation=True, max_length=MAX_TOKENS),
            sentence_weights):
        for prediction in predictions:
            weighted_scores[prediction["label"].lower()] += (
                prediction["score"] * weight / total_weight)

    baseline_labels = [p["label"].lower() for p in
                       baseline(sentences, truncation=True, max_length=MAX_TOKENS)]
    #max over a set breaks ties in set iteration order, which Python randomises
    #per process, so 376 of 3,526 turns changed label between two runs of the
    #same corpus. Counter keeps first-seen order and a tie goes to the label the
    #turn opened with, which is at least the same answer every time
    baseline_counts = Counter(baseline_labels)
    base_label = max(baseline_counts, key=baseline_counts.get)
    fin_label = max(weighted_scores, key=weighted_scores.get)

    return {
        "n_sentences": len(sentences),
        "n_tokens_max": max(n_tokens),
        "n_sentences_truncated": sum(1 for n in n_tokens if n > MAX_TOKENS),
        "finbert_positive": round(weighted_scores["positive"], 4),
        "finbert_negative": round(weighted_scores["negative"], 4),
        "finbert_neutral": round(weighted_scores["neutral"], 4),
        "finbert_label": fin_label,
        "baseline_label": base_label,
        # sst2 is binary and FinBERT is not
        # A neutral turn has nothing to agree with, and comparing it
        # would just measure the missing class
        "models_agree": None if fin_label == "neutral" else fin_label == base_label,
    }

This function is used to score one turn, averaging its sentences by length, so a long turn is not decided by its last line.


In [41]:
def build_sentiment(df_turns, batch=64):
    from transformers import AutoTokenizer, pipeline

    tokeniser = AutoTokenizer.from_pretrained(FINBERT)
    finbert = pipeline("text-classification", model=FINBERT, top_k=None, device=0)
    baseline = pipeline("text-classification", model=BASELINE, device=0)

    sentiment_rows = []
    for turn in tqdm(df_turns.itertuples(), total=len(df_turns), desc="scoring"):
        turn_scores = score_turn(turn.text, finbert, baseline, tokeniser)
        if turn_scores is None:
            continue
        sentiment_rows.append({"turn_id": turn.turn_id,
                               "doc_sha256": turn.doc_sha256, "bank": turn.bank,
                               "section": turn.section, "role": turn.role,
                               "n_words": int(turn.n_words), **turn_scores})
    return pd.DataFrame(sentiment_rows)

This function is used to score every turn with FinBERT and the baseline model.


In [42]:
def score_or_load_sentiment(df_turns, path=SENTIMENT_FILE,
                            rescore=RESCORE_SENTIMENT, run=RUN_FINBERT):
    if not run:
        print("RUN_FINBERT is False, so no turn was scored "
              "(set RUN_FINBERT=True to score them)")
        return None, False

    if os.path.isfile(path) and not rescore:
        df_cached = pd.read_parquet(path)
        print(f"loaded {path} "
              f"(delete it or set RESCORE_SENTIMENT=True to score again)")
        return df_cached, False

    df_sentiment = build_sentiment(df_turns)
    #saved here rather than by the caller, so the file the next run loads is
    #always the one this run produced
    write_table(df_sentiment, path)
    print(f"scored and saved {path}")
    return df_sentiment, True

This function is used to load the scored turns from disk, or score them if there is nothing saved yet. Scoring takes about ten minutes.


In [43]:
def build_exchange_sentiment(df_sentiment, df_exchanges, df_turns):
    #Aggregated from the turn scores, never re-scored, so the two tables cannot
    #disagree about the same text
    scores_by_turn = df_sentiment.set_index("turn_id")
    words_by_turn = df_turns.set_index("turn_id").n_words

    exchange_rows = []
    for exchange in df_exchanges.itertuples():
        answer_ids = [turn_id for turn_id in exchange.answer_turn_ids
                      if turn_id in scores_by_turn.index]
        if not answer_ids or exchange.question_turn_id not in scores_by_turn.index:
            continue
        answer_weights = [float(words_by_turn.get(turn_id, 1)) or 1.0
                          for turn_id in answer_ids]
        total_weight = sum(answer_weights) or 1.0
        answer_scores = {
            label: sum(float(scores_by_turn.at[turn_id, f"finbert_{label}"])
                       * weight / total_weight
                       for turn_id, weight in zip(answer_ids, answer_weights))
            for label in ("positive", "negative", "neutral")}
        question_scores = scores_by_turn.loc[exchange.question_turn_id]
        exchange_rows.append({
            "exchange_id": exchange.exchange_id,
            "doc_sha256": exchange.doc_sha256, "bank": exchange.bank,
            "period": exchange.period, "audience": exchange.audience,
            "analyst": exchange.analyst, "n_answer_turns": len(answer_ids),
            "q_n_words": int(exchange.n_words_q),
            "q_positive": round(float(question_scores.finbert_positive), 4),
            "q_negative": round(float(question_scores.finbert_negative), 4),
            "q_neutral": round(float(question_scores.finbert_neutral), 4),
            "q_label": question_scores.finbert_label,
            "a_n_words": int(exchange.n_words_a),
            "a_positive": round(answer_scores["positive"], 4),
            "a_negative": round(answer_scores["negative"], 4),
            "a_neutral": round(answer_scores["neutral"], 4),
            "a_label": max(answer_scores, key=answer_scores.get),
            #Negative means management was more downbeat than the analyst
            "tone_gap": round(answer_scores["positive"]
                              - float(question_scores.finbert_positive), 4),
        })
    return pd.DataFrame(exchange_rows)

This function is used to aggregate the turn scores up to the exchange, never re-scoring, so the two tables cannot disagree.


## Topics


In [44]:
def clean_question(text):
    without_boilerplate = BOILERPLATE.sub("", text or "")
    #Stripped to nothing means pure politeness, so keep the original
    without_boilerplate = " ".join(without_boilerplate.split())
    if len(without_boilerplate.split()) >= 5:
        return without_boilerplate
    return " ".join((text or "").split())

This function is used to strip the politeness off a question before it reaches the topic model.


In [45]:
def fit_or_load_bertopic(docs, path=MODEL_DIR, refit=REFIT_BERTOPIC, ids=None):
    from bertopic import BERTopic

    if os.path.isdir(path) and not refit:
        cached_model = BERTopic.load(path)
        print(f"loaded {path} (delete it or set REFIT_BERTOPIC=True to refit)")
        return cached_model, False

    from hdbscan import HDBSCAN
    from sentence_transformers import SentenceTransformer
    from sklearn.feature_extraction.text import CountVectorizer
    from umap import UMAP

    embedder = SentenceTransformer(EMBEDDINGS)
    #random_state pins UMAP, so the same corpus gives the same topic numbers
    reducer = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine",
                   random_state=SEED)
    #Replace the default eom with leaf
    #Analyst questions are lexically homogeneous, and eom put
    #1,122 of the 1,176 in one topic with no outliers at all
    #leaf splits the same tree lower down and finds 14
    clusterer = HDBSCAN(min_cluster_size=15, metric="euclidean",
                        cluster_selection_method="leaf", prediction_data=True)
    # Without a stoplist every topic is labelled "the, to, you, and"
    vectoriser = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))

    fitted_model = BERTopic(embedding_model=embedder, umap_model=reducer,
                            hdbscan_model=clusterer, vectorizer_model=vectoriser,
                            verbose=False)
    topic_ids = fitted_model.fit_transform(docs)[0]

    #leaf leaves about half unassigned. Reassigning to the nearest topic is a
    #documented BERTopic step, and the alternative is losing half the corpus
    topic_ids = fitted_model.reduce_outliers(docs, topic_ids, strategy="c-tf-idf")
    fitted_model.update_topics(docs, topics=topic_ids, vectorizer_model=vectoriser)
    fitted_model.save(path, serialization="safetensors", save_ctfidf=True,
                      save_embedding_model=EMBEDDINGS)
    #topics_ is kept by position only, so the ids let a reload re-align it
    if ids is not None:
        with open(os.path.join(path, FIT_IDS_FILE), "w", encoding="utf-8") as fh:
            json.dump([str(i) for i in ids], fh)
    print(f"fitted and saved {path}")
    return fitted_model, True

This function is used to load the fitted topic model from disk, or fit it if there is nothing saved yet.


In [46]:
def build_topics(df_exchanges, path=MODEL_DIR, refit=REFIT_BERTOPIC,
                 run=RUN_BERTOPIC):
    if not run:
        print("RUN_BERTOPIC is False, so no question was given a topic "
              "(set RUN_BERTOPIC=True to fit or load the model)")
        return None, None

    questions = [clean_question(q) for q in df_exchanges.question_text]
    model, _ = fit_or_load_bertopic(questions, path, refit,
                                    ids=df_exchanges.exchange_id)
    transform_ids, probabilities = model.transform(questions)
    #transform gives one score per document
    #Or a row per topic, if the model had calculate_probabilities
    probabilities = ([float(max(p) if hasattr(p, "__len__") else p)
                      for p in probabilities] if probabilities is not None
                     else [float("nan")] * len(questions))

    #transform undoes reduce_outliers: straight after a fit it reruns HDBSCAN
    #and left 508 of 1,168 in -1, and after a reload (safetensors keeps no
    #HDBSCAN) it matches by embedding and agreed on 31. So a fitted question
    #keeps its reduced topic, matched by exchange_id rather than position
    fit_ids_path = os.path.join(path, FIT_IDS_FILE)
    fitted_topic_by_id = {}
    if os.path.isfile(fit_ids_path):
        fit_ids = json.load(open(fit_ids_path, encoding="utf-8"))
        fitted_topic_by_id = dict(zip(fit_ids, model.topics_))
    topic_ids, topic_probabilities = [], []
    for exchange_id, transform_id, p in zip(df_exchanges.exchange_id,
                                            transform_ids, probabilities):
        if exchange_id in fitted_topic_by_id:
            #No per question score survives the save, so none is invented
            topic_ids.append(int(fitted_topic_by_id[exchange_id]))
            topic_probabilities.append(float("nan"))
        else:
            topic_ids.append(int(transform_id))
            topic_probabilities.append(round(p, 4))
    n_from_fit = sum(1 for i in df_exchanges.exchange_id if i in fitted_topic_by_id)
    print(f"{n_from_fit} questions kept their fitted topic, "
          f"{len(topic_ids) - n_from_fit} assigned by transform "
          f"(no {FIT_IDS_FILE} means every question is)")
    topic_info = model.get_topic_info().set_index("Topic")

    df_topics = pd.DataFrame({
        "exchange_id": df_exchanges.exchange_id.values,
        "doc_sha256": df_exchanges.doc_sha256.values,
        "bank": df_exchanges.bank.values,
        "period": df_exchanges.period.values,
        "audience": df_exchanges.audience.values,
        "analyst": df_exchanges.analyst.values,
        "topic_id": topic_ids,
        "topic_label": [topic_info.Name.get(t, "?") for t in topic_ids],
        "probability": topic_probabilities,
        "question_cleaned": questions,
    })
    df_topic_info = pd.DataFrame({
        "topic_id": topic_info.index,
        "label": topic_info.Name.values,
        "size": topic_info.Count.values,
        "top_words": ["; ".join(word for word, _ in (model.get_topic(t) or [])[:10])
                      for t in topic_info.index],
    })
    return df_topics, df_topic_info

This function is used to assign a topic to every analyst question and return the topic list alongside it.


## Quality gates


In [47]:
def qa_report(df_turns, df_exchanges, df_documents, manifest_path=None):
    gate_rows = []

    def gate(name, expected, actual, passed):
        gate_rows.append({"Gate": name, "Expected": expected, "Actual": actual,
                     "Pass": bool(passed)})

    n_failed_documents = int((df_documents.parse_status == "failed").sum())
    gate("1. every document parses", "0 failed", f"{n_failed_documents} failed",
         n_failed_documents == 0)

    gate("2. enough turns", ">= 3000", f"{len(df_turns)}", len(df_turns) >= 3000)
    gate("2. enough exchanges", ">= 1400", f"{len(df_exchanges)}",
         len(df_exchanges) >= 1400)

    expects_turns = df_documents[df_documents.doc_format != "prepared_only"]
    short_qa_documents = expects_turns[expects_turns.n_turns < 5]
    gate("3. every Q&A document yields >= 5 turns", "0 short",
         f"{len(short_qa_documents)} short"
         + (f": {list(short_qa_documents.relpath)[:3]}" if len(short_qa_documents) else ""),
         len(short_qa_documents) == 0)

    # The pattern matches "deficiencies" too, so the dictionary decides
    corrupted_words = []
    for turn_text in df_turns.text:
        corrupted_words += [w for w in SILENT_CORRUPTION.findall(turn_text)
                            if not is_word(w)]
    gate("4. no silent ligature corruption", "0 matches",
         f"{len(corrupted_words)} matches"
         + (f": {sorted(set(corrupted_words))[:4]}" if corrupted_words else ""),
         not corrupted_words)

    if manifest_path:
        with open(manifest_path, encoding="utf-8") as fh:
            manifest_hashes = {r["sha256"] for r in csv.DictReader(fh)}
        orphan_hashes = set(df_turns.doc_sha256) - manifest_hashes
        gate("5. every turn joins to the manifest", "0 orphans",
             f"{len(orphan_hashes)} orphans", not orphan_hashes)

    qa_turns = df_turns[df_turns.section == "qa"]
    for bank, bank_qa_turns in qa_turns.groupby("bank"):
        analyst_share = (bank_qa_turns.role == "analyst").mean()
        gate(f"6. analyst share of Q&A turns ({bank})", "35% to 65%",
             f"{analyst_share:.0%}", 0.35 <= analyst_share <= 0.65)

    n_empty_turns = int((df_turns.n_words < 1).sum())
    monologue = set(df_documents[df_documents.doc_format.isin(MONOLOGUE_FORMATS)]
                    .doc_sha256)
    runaway_turns = df_turns[(df_turns.n_words > MAX_TURN_WORDS)
                             & ~df_turns.doc_sha256.isin(monologue)]
    gate("7. no empty turn", "0", f"{n_empty_turns}", n_empty_turns == 0)
    gate(f"7. no turn over {MAX_TURN_WORDS} words", "0",
         f"{len(runaway_turns)}" + (f", longest {int(df_turns.n_words.max())}"
                                    if len(runaway_turns) else ""),
         len(runaway_turns) == 0)

    return pd.DataFrame(gate_rows)

This function is used to run the ten acceptance gates on the three tables and return them as a table, naming the documents that fail rather than reporting an aggregate.


In [48]:
def is_list_column(values):
    return values.map(lambda v: isinstance(v, (list, tuple, np.ndarray))).any()

This function is used to spot the columns holding lists, which cannot be counted or compared like the rest.


In [49]:
def describe_table(df_table, name):
    rows = []
    for column in df_table.columns:
        values = df_table[column]
        #lists are unhashable, so answer_turn_ids and responder_names would
        #raise on nunique. They are counted as present or not instead
        listy = is_list_column(values)
        blank = values.isnull() | (values.astype(str).str.strip() == "")
        rows.append({
            "table": name,
            "column": column,
            "dtype": str(values.dtype),
            "n_blank": int(blank.sum()),
            "n_unique": None if listy else int(values.nunique()),
            "example": str(values.iloc[0])[:40] if len(values) else "",
        })
    df_described = pd.DataFrame(rows)
    #Int64 rather than the float pandas picks when a column holds None, so a
    #list column reads as <NA> instead of NaN next to the real counts
    df_described["n_unique"] = df_described.n_unique.astype("Int64")
    return df_described

This function is used to check a new table at a glance, one row per column, counting blanks and unique values.


In [50]:
#Extension of the previous function to every table at once, which is the shape
#check a consumer wants before joining any of them together
def compare_tables(tables, keys=None):
    keys = keys or {}
    rows = []
    for name, df_table in tables.items():
        key = keys.get(name)
        #exchanges carries answer_turn_ids as a list, and .duplicated() raises on
        #an unhashable column, so the list columns sit the row comparison out
        hashable = [c for c in df_table.columns
                    if not is_list_column(df_table[c])]
        rows.append({
            "table": name,
            "n_rows": len(df_table),
            "n_columns": len(df_table.columns),
            "n_duplicate_rows": int(df_table[hashable].duplicated().sum()),
            "key": key or "",
            "n_duplicate_keys": (int(df_table[key].duplicated().sum())
                                 if key and key in df_table.columns else 0),
            "memory_mb": round(df_table.memory_usage(deep=True).sum() / 1e6, 1),
        })
    return pd.DataFrame(rows)

Extension of the previous function to every table at once, which is what a consumer wants before joining them together.


In [51]:
def inspect_parse_failures(df_documents, df_turns, n=5):
    failed_documents = df_documents[df_documents.parse_status != "ok"]
    if failed_documents.empty:
        print("every document parsed cleanly")
        return failed_documents

    print(f"{len(failed_documents)} document(s) did not parse cleanly:\n")
    for document in failed_documents.head(n).itertuples():
        print(f"  {document.parse_status:10s} {document.relpath}")
        print(f"             {document.n_pages} pages, {document.n_turns} turns, "
              f"{document.n_exchanges} exchanges, format {document.doc_format}")
        if document.warnings:
            print(f"             warning: {document.warnings}")
        document_turns = df_turns[df_turns.doc_sha256 == document.doc_sha256].text
        first_turn_text = (document_turns.iloc[0][:200] if len(document_turns)
                           else "(no turns at all)")
        print(f"             first 200 chars: {first_turn_text}\n")
    return failed_documents

This function is used to look at the documents that did not parse cleanly and print the start of what they produced.


In [52]:
def qa_report_models(df_turns, df_sentiment, df_topics, df_topic_info,
                     model_dir=MODEL_DIR):
    gate_rows = []

    def gate(name, expected, actual, passed):
        gate_rows.append({"Gate": name, "Expected": expected, "Actual": actual,
                          "Pass": bool(passed)})

    unscored_turns = set(df_turns.turn_id) - set(df_sentiment.turn_id)
    gate("8. every turn has a sentiment row", "0 missing",
         f"{len(unscored_turns)} missing", not unscored_turns)
    n_duplicate_scores = int(df_sentiment.turn_id.duplicated().sum())
    gate("8. exactly one row per turn", "0 duplicates",
         f"{n_duplicate_scores} duplicates", n_duplicate_scores == 0)

    truncated = df_sentiment[df_sentiment.n_sentences_truncated > 0]
    gate("9. no turn scored from truncated input", "0 turns",
         f"{len(truncated)} turns" + (f": {list(truncated.turn_id)[:3]}"
                                      if len(truncated) else ""),
         len(truncated) == 0)

    outlier_share = (df_topics.topic_id == -1).mean() if len(df_topics) else 1.0
    gate("10. BERTopic outlier share", "< 40%", f"{outlier_share:.0%}",
         outlier_share < 0.40)

    real_topics = df_topic_info[df_topic_info.topic_id != -1]
    biggest_topic = real_topics.sort_values("size", ascending=False).head(1)
    biggest_topic_words = (biggest_topic.top_words.iloc[0] if len(biggest_topic)
                           else "")
    #A top topic of "thanks for taking my question" means the stoplist failed
    boilerplate = {"thanks", "thank", "morning", "okay", "great", "helpful"}
    top_is_boilerplate = bool(
        boilerplate & set(biggest_topic_words.replace(";", " ").split()[:4]))
    gate("10. largest topic is not boilerplate", "not boilerplate",
         biggest_topic_words[:44] or "(no topics)",
         not top_is_boilerplate and len(real_topics) > 0)

    gate("11. the fitted model was saved", "exists on disk",
         "yes" if os.path.isdir(model_dir) else "no", os.path.isdir(model_dir))

    return pd.DataFrame(gate_rows)

This function is used to run the six gates that only make sense once the sentiment and topic models have run.


# Driver

The build runs from here, using the functions above.


## The corpus


In [53]:
df_turns, df_exchanges, df_documents, built = build_or_load_corpus()

loaded the tables from /home/jintoski/projects/CAM_DS_BOE_T1/employer_project/corpus (set REBUILD_CORPUS=True to build them again)


104 transcripts over 1,378 pages, giving 3,560 turns and 1,262 exchanges. 102 of the documents parse cleanly, one is a three page call holding two turns, and one has no Q&A in it at all.


## What the tables hold


In [54]:
compare_tables({"turns": df_turns, "exchanges": df_exchanges,
                "documents": df_documents},
               {"turns": "turn_id", "exchanges": "exchange_id",
                "documents": "doc_sha256"})

,table,n_rows,n_columns,n_duplicate_rows,key,n_duplicate_keys,memory_mb
0,turns,3560,18,0,turn_id,0,8.2
1,exchanges,1262,16,0,exchange_id,0,5.3
2,documents,104,18,0,doc_sha256,0,0.1


The three tables side by side. No duplicate keys in any of them. answer_turn_ids sits the uniqueness checks out, because a list column cannot be counted that way.


In [55]:
describe_table(df_turns, "turns")

,table,column,dtype,n_blank,n_unique,example
0,turns,turn_id,object,0,3560,5c9f6bef592a-0000
1,turns,doc_sha256,object,0,104,5c9f6bef592afd9681b49057fb7b07ce3d1971a8
2,turns,bank,object,0,2,barclays
3,turns,fiscal_year,int64,0,6,2021
4,turns,fiscal_quarter,int64,0,4,1
5,turns,audience,object,0,2,equity
6,turns,doc_format,object,0,3,qa_only
7,turns,section,object,0,2,qa
8,turns,speaker,object,0,145,Joseph Dickerson
9,turns,speaker_norm,object,0,145,joseph_dickerson


3,560 turns over 18 columns, 1,850 of them Barclays and 1,710 HSBC.


In [56]:
describe_table(df_exchanges, "exchanges")

,table,column,dtype,n_blank,n_unique,example
0,exchanges,exchange_id,object,0,1262,5c9f6bef592a-x000
1,exchanges,doc_sha256,object,0,93,5c9f6bef592afd9681b49057fb7b07ce3d1971a8
2,exchanges,bank,object,0,2,barclays
3,exchanges,period,object,0,22,2021_Q1
4,exchanges,audience,object,0,2,equity
5,exchanges,analyst,object,0,112,Joseph Dickerson
6,exchanges,analyst_firm,object,0,42,Jefferies
7,exchanges,question_text,object,0,1250,My two questions are the following. Firs
8,exchanges,question_turn_id,object,0,1262,5c9f6bef592a-0000
9,exchanges,question_page,int64,0,21,1


1,262 exchanges, one for every analyst question that got an answer.


## Gates


In [57]:
df_gates = qa_report(df_turns, df_exchanges, df_documents, MANIFEST)
df_gates

,Gate,Expected,Actual,Pass
0,1. every document parses,0 failed,0 failed,True
1,2. enough turns,>= 3000,3560,True
2,2. enough exchanges,>= 1400,1262,False
3,3. every Q&A document yields >= 5 turns,0 short,1 short: ['barclays/2024_Q2/transcript__fixed-...,False
4,4. no silent ligature corruption,0 matches,0 matches,True
5,5. every turn joins to the manifest,0 orphans,0 orphans,True
6,6. analyst share of Q&A turns (barclays),35% to 65%,40%,True
7,6. analyst share of Q&A turns (hsbc),35% to 65%,36%,True
8,7. no empty turn,0,0,True
9,7. no turn over 5000 words,0,0,True


Eight of the ten gates pass, and the two that fail have been left failing rather than having their thresholds moved. The corpus holds 1,262 exchanges against a target of 1,400. The Barclays 2024 Q2 fixed income call yields 2 turns against a minimum of 5, because it is a three page document in which one analyst asks one question. Neither is a parsing defect.


In [58]:
inspect_parse_failures(df_documents, df_turns)

2 document(s) did not parse cleanly:

  low_turns  barclays/2024_Q2/transcript__fixed-income-q-and-a-transcript-pdf-138kb.pdf
             3 pages, 2 turns, 1 exchanges, format qa_only
             first 200 chars: I have three please. Starting off where you finished on risk transfers. These obviously seems to have come to a lot more prominence recently. So my question is, is this just a risk management tool for

  no_qa      hsbc/2021_Q2/transcript__interim-results-2021-presentation-to-fixed-income-investors-transcript.pdf
             7 pages, 15 turns, 0 exchanges, format prepared_and_qa
             first 200 chars: Hi, everyone. This is Carlo. I'm joined today by Richard O'Connor, our head of Investor Relations; and Greg Case, Head of Debt Investor Relations; together with Richard Boyns, Head of Capital Manageme



,doc_sha256,relpath,bank,fiscal_year,fiscal_quarter,audience,doc_format,source_url,source_page,fetched_at,n_pages,n_turns,n_exchanges,n_speakers,fffd_before,silent_repairs,parse_status,warnings
35,ef4c3b9c9bdb48e06ec24c146095b9b6be5d24728b119a...,barclays/2024_Q2/transcript__fixed-income-q-an...,barclays,2024,2,fixed_income,qa_only,https://home.barclays/content/dam/home-barclay...,https://home.barclays/investor-relations/repor...,2026-08-31T14:50:11+00:00,3,2,1,2,0,0,low_turns,
64,1635834082a777a917acd712fb8114fe1727199b6352fb...,hsbc/2021_Q2/transcript__interim-results-2021-...,hsbc,2021,2,fixed_income,prepared_and_qa,https://www.hsbc.com/-/files/hsbc/investors/hs...,https://www.hsbc.com/investors/results-and-ann...,2026-08-31T15:00:31+00:00,7,15,0,3,0,0,no_qa,


The two documents worth looking at by eye, printed with the first 200 characters each of them produced.


## Sentiment


In [59]:
if RUN_FINBERT:
    df_sentiment, scored = score_or_load_sentiment(df_turns)
    print(df_sentiment.finbert_label.value_counts())
else:
    df_sentiment = None
    print("RUN_FINBERT is False, so the tone scores were skipped")

loaded /home/jintoski/projects/CAM_DS_BOE_T1/employer_project/corpus/sentiment.parquet (delete it or set RESCORE_SENTIMENT=True to score again)
finbert_label
neutral     2973
positive     485
negative     102
Name: count, dtype: int64


With RUN_FINBERT set to True this scores every turn, which takes about ten minutes the first time and is read back from disk after that. The last full run read 2,973 turns neutral, 485 positive and 102 negative, with FinBERT and the general model agreeing on 78 of every 100 turns that are not neutral. With the flag False nothing is scored and nothing is written.


In [60]:
if RUN_FINBERT:
    df_exchange_sentiment = build_exchange_sentiment(df_sentiment,
                                                    df_exchanges, df_turns)
    write_table(df_exchange_sentiment, SENTIMENT_EXCHANGES_FILE)
else:
    print("RUN_FINBERT is False, so there is nothing to roll up")

The same scores rolled up to the 1,262 exchanges, aggregated from the turn scores rather than scored again, so the two tables cannot disagree. Skipped with the flag False, because it has no turn scores to read.


## Topics


In [61]:
if RUN_BERTOPIC:
    df_topics, df_topic_info = build_topics(df_exchanges)
    write_table(df_topics, TOPICS_FILE)
    write_table(df_topic_info, TOPIC_INFO_FILE)
    print(df_topic_info)
else:
    df_topics = df_topic_info = None
    print("RUN_BERTOPIC is False, so no topics were assigned")

2026-10-08 23:25:16.534117: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-08 23:25:16.644877: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1791494716.672111   64172 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1791494716.682785   64172 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1791494716.785728   64172 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

loaded /home/jintoski/projects/CAM_DS_BOE_T1/employer_project/corpus/topic_model (delete it or set REFIT_BERTOPIC=True to refit)


1262 questions kept their fitted topic, 0 assigned by transform (no fit_exchange_ids.json means every question is)
    topic_id                            label  size  \
0          0       0_hong kong_hong_kong_just   183   
1          1    1_okay_great_helpful_consider    74   
2          2           2_hedge_just_rate_year   164   
3          3         3_just_cards_uk_business   156   
4          4      4_rwas_think_rwa_investment   104   
5          5         5_mortgage_uk_just_rates    79   
6          6    6_just_sorry just_sorry_right    48   
7          7            7_just_tier_ve_legacy    62   
8          8           8_cost_costs_year_just    79   
9          9  9_capital_buybacks_just_buyback    71   
10        10   10_deposits_deposit_just_rates    50   
11        11          11_nim_quarter_nii_just    84   
12        12           12_just_think_cost_bit   108   

                                            top_words  
0   hong kong; hong; kong; just; growth; wealth; q...  
1 

With RUN_BERTOPIC set to True this loads the fitted model and gives each question a topic, 13 of them over the 1,262 questions with no outliers left once reduce_outliers has run. With the flag False no model is loaded or fitted and no topic table is written.


## Model gates


In [62]:
if RUN_FINBERT and RUN_BERTOPIC:
    df_model_gates = qa_report_models(df_turns, df_sentiment, df_topics,
                                      df_topic_info)
    print(df_model_gates)
else:
    print("The model gates need both RUN_FINBERT and RUN_BERTOPIC")

                                     Gate         Expected  \
0       8. every turn has a sentiment row        0 missing   
1             8. exactly one row per turn     0 duplicates   
2  9. no turn scored from truncated input          0 turns   
3              10. BERTopic outlier share            < 40%   
4    10. largest topic is not boilerplate  not boilerplate   
5          11. the fitted model was saved   exists on disk   

                                         Actual  Pass  
0                                     0 missing  True  
1                                  0 duplicates  True  
2                                       0 turns  True  
3                                            0%  True  
4  hong kong; hong; kong; just; growth; wealth;  True  
5                                           yes  True  


These six gates read the sentiment and topic tables, so they need both flags on. The last run with both on passed all six, making 14 of the 16 gates overall.
